In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors

# fastlisaresponse must be imported before few (avoids crash due to library init order)
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits

from WDMWaveletTransforms.wavelet_transforms import transform_wavelet_time
from few.trajectory.inspiral import EMRIInspiral, get_fundamental_frequencies, MTSUN_SI
from few.waveform import GenerateEMRIWaveform

# Params from ToMajo.txt
M  = 1134944.869275098
m  = 29.489999547765798
a  = 0.5
p0 = 10.0
e0 = 0.22865665220266215
Y0 = 1.0
Phi_phi0, Phi_theta0, Phi_r0 = 0., 1.2, 3.5
qS, phiS, qK, phiK = 0.2, 0.2, 0.8, 0.8
dist = 5.235888314207546
dt = 10.0
T  = 1.0

In [ ]:
# FEW waveform: single (2,2,0,0) mode, KerrEccEq
wave_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, output_type='td', odd_len=True),
    mode_selector_kwargs={'mode_selection': [(2, 2, 0, 0)]},
    Ylm_kwargs={'include_minus_m': True},
    inspiral_kwargs={'err': 1e-10, 'dense_stepping': 0},
)

h = np.real(wave_gen(
    M, m, a, p0, e0, Y0, dist,
    qS, phiS, qK, phiK,
    Phi_phi0, Phi_theta0, Phi_r0,
    dt=dt, T=T
)).astype(np.float64)

# KerrEccEqFlux trajectory for frequency track
traj = EMRIInspiral(func='KerrEccEqFlux')
t, p, e, x, _, _, _ = traj(
    M, m, a, p0, e0, Y0,
    Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
    dt=dt, T=T, upsample=True, fix_t=True
)
OmegaPhi, _, _ = get_fundamental_frequencies(a, p, e, x)
f_220 = 2 * OmegaPhi / (M * MTSUN_SI) / (2 * np.pi)

print(f'h: {len(h)} samples, f_220: {f_220.min():.4e} - {f_220.max():.4e} Hz')

In [ ]:
def scalogram(ax, data, dt, Nt=64, vmin=1e-6):
    Nf = len(data) // Nt
    ND = Nf * Nt
    wt = np.linspace(0, ND * dt, Nt + 1)
    wf = np.arange(0, Nf + 1) / (2 * dt * Nf)
    aa = np.abs(transform_wavelet_time(data[:ND], Nf, Nt))  # shape (Nt, Nf)
    aa /= aa.max()
    aa = np.clip(aa, vmin, 1.0)
    # pcolormesh needs (Nf, Nt) grid for (Nt+1) x (Nf+1) edges
    ax.pcolormesh(wt, wf, aa.T,
                  cmap='inferno', norm=colors.LogNorm(vmin=vmin, vmax=1.0),
                  shading='auto', rasterized=True)
    return wt, wf

fig, ax = plt.subplots(figsize=(12, 5))
wt, wf = scalogram(ax, h, dt)
ax.plot(t, f_220, color='cyan', lw=1.5, label=r'$2f_\phi$ — (2,2,0,0)')
ax.set_ylim(wf[1], 0.005)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Frequency [Hz]')
ax.set_title('FEW (2,2,0,0) mode: WDM + frequency track')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TDI response (same config as kerr_map.py)
resp = ResponseWrapper(
    wave_gen, T, dt,
    index_lambda=8, index_beta=7,
    flip_hx=True, remove_sky_coords=False,
    is_ecliptic_latitude=False, remove_garbage=True,
    t0=100000., order=25,
    tdi='1st generation', tdi_chan='AET',
    orbits=EqualArmlengthOrbits(),
)

A, E, Tch = resp(
    M, m, a, p0, e0, Y0, dist,
    qS, phiS, qK, phiK,
    Phi_phi0, Phi_theta0, Phi_r0,
)

# Align frequency track to trimmed TDI time
t_tdi = np.arange(len(A)) * dt
t_offset = t[-1] - t_tdi[-1]
mask = t >= t_offset

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, chan, label in zip(axes, [A, E], ['A', 'E']):
    _, wf_tdi = scalogram(ax, chan, dt)
    ax.plot(t[mask] - t_offset, f_220[mask], color='cyan', lw=1.5,
            label=r'$2f_\phi$ — (2,2,0,0)')
    ax.set_ylim(wf_tdi[1], 0.005)
    ax.set_xlabel('Time [s]')
    ax.set_title(f'TDI {label} — (2,2,0,0) mode')
    ax.legend()
axes[0].set_ylabel('Frequency [Hz]')
plt.tight_layout()
plt.show()